# COD-21: 2025 environmental-exposure overlay

This notebook reproduces the ambient particulate and B-IMD overlap without rewriting tracked artifacts. It does not attribute exposure to ports or industrial complexes.

In [ ]:
import os
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    path for path in (current, *current.parents)
    if (path / 'pyproject.toml').is_file()
)
os.chdir(project_root)
project_root

In [ ]:
import json

import pandas as pd
import plotly.express as px

from busan_imd.environmental_overlay import DEFAULT_COMPOSITE, DEFAULT_PROFILE, build

composite = pd.read_csv(DEFAULT_COMPOSITE, dtype={'admin_dong_code': str})
profile = pd.read_csv(DEFAULT_PROFILE, dtype={'admin_dong_code': str})
overlay, report = build(composite, profile)
report['category_counts'], report['decision']

In [ ]:
category_order = ['double_burden', 'social_priority_only', 'high_air_only', 'neither']
scatter = px.scatter(
    overlay,
    x='b_imd_score_0_100',
    y='particulate_exposure_score_0_100',
    color='overlay_category',
    category_orders={'overlay_category': category_order},
    hover_data=['sigungu_name', 'admin_dong_name', 'b_imd_rank', 'particulate_exposure_rank'],
    title='B-IMD and ambient particulate exposure',
)
high_exposure_cutoff = overlay.loc[
    overlay['particulate_exposure_rank'] == 52,
    'particulate_exposure_score_0_100',
].iloc[0]
scatter.add_hline(
    y=high_exposure_cutoff, line_dash='dash', line_color='gray'
)
scatter.show()

In [ ]:
boundary_path = Path(
    'data/raw/sgis/admin_boundaries/2025/'
    'busan_admin_dong_boundaries_2025_valid.geojson'
)
boundaries = json.loads(boundary_path.read_text(encoding='utf-8'))
overlay_map = px.choropleth_mapbox(
    overlay,
    geojson=boundaries,
    locations='admin_dong_code',
    featureidkey='properties.adm_cd',
    color='overlay_category',
    category_orders={'overlay_category': category_order},
    hover_data=['sigungu_name', 'admin_dong_name', 'b_imd_rank', 'particulate_exposure_rank'],
    mapbox_style='carto-positron',
    center={'lat': 35.18, 'lon': 129.07},
    zoom=8.5,
    opacity=0.65,
    title='2025 ambient-air and social-vulnerability overlay',
)
overlay_map.update_layout(margin={'r': 0, 't': 45, 'l': 0, 'b': 0})
overlay_map.show()

In [ ]:
overlay.loc[overlay['double_burden'], [
    'b_imd_rank', 'sigungu_name', 'admin_dong_name',
    'particulate_exposure_rank', 'annual_pm25_ug_m3_idw_2025',
    'annual_pm10_ug_m3_idw_2025',
]].sort_values('b_imd_rank')

## Interpretation limit

The project has no versioned 2025 port or industrial-complex geometry. Treat these results only as ambient particulate double-burden screening; do not infer a source or causal effect.